In [0]:
import time
from pyspark.sql.functions import current_timestamp, lit, year, month, dayofmonth

In [0]:
storage_account_name = dbutils.secrets.get(scope="kv-finbank", key="datalake-account-name")
storage_account_access_key = dbutils.secrets.get(scope="kv-finbank", key="datalake-access-key")
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_access_key
)
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/"

In [0]:
landing_path = f"{bronze_path}landing/" 
delta_base_path = f"{bronze_path}delta/"
dbutils.widgets.text("batch_id", "MANUAL_RUN_001") 
batch_id = dbutils.widgets.get("batch_id")
errores_pipeline = []

def procesar_a_bronze(nombre_tabla, source_path, destination_path):
    '''Procesamiento de los datos para carga inicial y almacenamiento a capa bronze por tabla '''
    print(f"--- Iniciando ingesta para {nombre_tabla} ---")
    start_time = time.time()
    
    try:
        df = spark.read.parquet(source_path)
        df_audited = df \
            .withColumn("ingest_timestamp", current_timestamp()) \
            .withColumn("source_system", lit("SQL_FinBank_Core")) \
            .withColumn("batch_id", lit(batch_id))
            
        df_partitioned = df_audited \
            .withColumn("ingest_year", year("ingest_timestamp")) \
            .withColumn("ingest_month", month("ingest_timestamp")) \
            .withColumn("ingest_day", dayofmonth("ingest_timestamp"))
            
        df_partitioned.write \
            .format("delta") \
            .mode("overwrite") \
            .partitionBy("ingest_year", "ingest_month", "ingest_day") \
            .save(destination_path)
            
        registros = df_partitioned.count()
        duracion = round(time.time() - start_time, 2)
        print(f"LOG: Tabla {nombre_tabla} | Registros: {registros} | Duración: {duracion} seg | Modo: Overwrite")

    except Exception as e:
        error_msg = f"Error procesando {nombre_tabla}: {str(e)}"
        print(f"ERROR:{error_msg}")
        errores_pipeline.append(error_msg)

# --- Ejecución para nuestras tablas ---
tablas = [
    "TB_CLIENTES_CORE", "TB_MOV_FINANCIEROS", "TB_PRODUCTOS_CAT", 
    "TB_SUCURSALES_RED", "TB_OBLIGACIONES", "TB_COMISIONES_LOG"
]

for tabla in tablas:
    ruta_origen = f"{landing_path}{tabla}.parquet" 
    ruta_destino = f"{delta_base_path}{tabla}" 
    procesar_a_bronze(tabla, ruta_origen, ruta_destino)

if len(errores_pipeline) > 0:
    dbutils.notebook.exit(f"FAILED: Excepciones capturadas en: {errores_pipeline}")
else:
    dbutils.notebook.exit("SUCCESS")

--- Iniciando ingesta para TB_CLIENTES_CORE ---
LOG: Tabla TB_CLIENTES_CORE | Registros: 10000 | Duración: 30.13 seg | Modo: Incremental (Append)
--- Iniciando ingesta para TB_MOV_FINANCIEROS ---
LOG: Tabla TB_MOV_FINANCIEROS | Registros: 500000 | Duración: 13.3 seg | Modo: Incremental (Append)
--- Iniciando ingesta para TB_PRODUCTOS_CAT ---
LOG: Tabla TB_PRODUCTOS_CAT | Registros: 50 | Duración: 4.72 seg | Modo: Incremental (Append)
--- Iniciando ingesta para TB_SUCURSALES_RED ---
LOG: Tabla TB_SUCURSALES_RED | Registros: 200 | Duración: 4.83 seg | Modo: Incremental (Append)
--- Iniciando ingesta para TB_OBLIGACIONES ---
LOG: Tabla TB_OBLIGACIONES | Registros: 30000 | Duración: 5.18 seg | Modo: Incremental (Append)
--- Iniciando ingesta para TB_COMISIONES_LOG ---
LOG: Tabla TB_COMISIONES_LOG | Registros: 80000 | Duración: 5.38 seg | Modo: Incremental (Append)
